# Solution B2 · Removing noise from images with PCA

**Type**: Implement with scikit-learn

## Objective

Add Gaussian noise to the digits, reconstruct with `k` components and measure the error with respect to the **clean** image; choose `k` with a criterion that uses only the noisy data (the Marchenko-Pastur edge: the components with an eigenvalue above the expected noise) and
compare with the absolute best `k`; finally verify that a classifier trained on the components is more robust than one on the pixels.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline

X, y = load_digits(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, stratify=y, random_state=0)
sigma = 4
rng = np.random.default_rng(0)
X_tr_n = X_tr + rng.normal(0, sigma, X_tr.shape)
X_te_n = X_te + rng.normal(0, sigma, X_te.shape)
ks = [2, 5, 10, 15, 20, 25, 30, 40, 64]

## Commented solution

In [ ]:
noisy_mse = ((X_te_n - X_te) ** 2).mean()
mse_per_k = {}
for k in ks:
    p = PCA(k).fit(X_tr_n)                                            # PCA sees only noisy data
    reconstructed = p.inverse_transform(p.transform(X_te_n))
    mse_per_k[k] = ((reconstructed - X_te) ** 2).mean()                # but the error is measured against the clean image
mse_per_k = pd.Series(mse_per_k)
oracle_k = int(mse_per_k.idxmin())
print(f"error of the noisy image: {noisy_mse:.2f}")
print(mse_per_k.round(2).to_string())

In [ ]:
n, p_ = X_tr_n.shape
threshold = sigma ** 2 * (1 + np.sqrt(p_ / n)) ** 2                       # maximum eigenvalue expected if there were only noise
full = PCA().fit(X_tr_n)
mp_k = int((full.explained_variance_ > threshold).sum())
pk = PCA(mp_k).fit(X_tr_n)
mp_mse = ((pk.inverse_transform(pk.transform(X_te_n)) - X_te) ** 2).mean()
reduction = 1 - mse_per_k[oracle_k] / noisy_mse
print(f"noise threshold: {threshold:.1f} | components above the threshold: {mp_k} | error with those: {mp_mse:.2f} (oracle k={oracle_k}: {mse_per_k[oracle_k]:.2f}) | error reduction with the oracle: {reduction:.0%}")

In [ ]:
pixel_acc = LogisticRegression(max_iter=3000).fit(X_tr_n, y_tr).score(X_te_n, y_te)
pca_acc = make_pipeline(PCA(oracle_k), LogisticRegression(max_iter=3000)).fit(X_tr_n, y_tr).score(X_te_n, y_te)
clean_acc = LogisticRegression(max_iter=3000).fit(X_tr, y_tr).score(X_te, y_te)
print(f"logistic on noisy pixels {pixel_acc:.3f} | on PCA({oracle_k}) of noisy data {pca_acc:.3f} | on clean data {clean_acc:.3f}")

## Check

In [ ]:
with verify():
    check(close(noisy_mse, ((X_te_n - X_te) ** 2).mean()) and close(noisy_mse, 16, rtol=0.1), "the noisy image has an error of about 16 (= sigma^2)", "np.mean((X_te_n - X_te) ** 2)")
    check(list(mse_per_k.index) == ks and mse_per_k[64] > 0.97 * noisy_mse, "with 64 components (no filter) the error stays that of the noise", "reconstruction with all the components = noisy image")
    check(oracle_k == mse_per_k.idxmin() and 10 <= oracle_k <= 20 and mse_per_k[oracle_k] < 0.5 * noisy_mse, "the best k is between 10 and 20 and halves the error (over 50%)", "idxmin of mse_per_k")
    check(20 <= mp_k <= 35 and mp_mse < 0.6 * noisy_mse and mp_mse > mse_per_k[oracle_k], "the noise-edge criterion gives k between 20 and 35: almost optimal, but more conservative than the oracle", "components with an eigenvalue above the threshold")
    check(close(reduction, 1 - mse_per_k[oracle_k] / noisy_mse) and reduction > 0.5, "the error is reduced by over 50%", "1 - oracle_mse / noisy_mse")
    check(pca_acc > pixel_acc + 0.02 and clean_acc > 0.95 and pixel_acc < 0.9, "on the noisy images the logistic on the components beats the one on the pixels by over 2 points (clean: above 0.95)", "compare pca_acc, pixel_acc, clean_acc")
done()

## What you learned

- **Reconstruction with few components filters the noise:** with 15 components the error with respect to the clean image drops from 16 to 7.2 (−55%), better than the noisy image and much better than the 64 pixels. With too many components (64) the error is that of the noise; with too few
  (2) the signal is lost (14).
- The tradeoff is bias-variance applied to the reconstruction: every added component brings a bit of signal and a quantity of noise. The minimum is where the marginal signal equals the noise.
- **Without the clean image** you can choose `k` with the Marchenko-Pastur edge: the components with an eigenvalue above $\sigma^2(1+\sqrt{p/n})^2$ are not compatible with noise alone. Here it indicates 25 (error 8.0), more conservative than the optimum (15, error 7.2) but almost as good and
  chosen **only with the noisy data** and knowledge of `sigma`.
- The filter also helps a classifier trained on the noisy data (0.86 against 0.82 on the pixels): the advantage is modest compared to the clean case (0.97), because the noise is strong, but it is free. It is the reason why PCA appears in so many pipelines on sensor data and images.

## Common pitfall

Choosing `k` by minimizing the reconstruction error **on the noisy training set**: it always decreases with `k` and would choose 64, that is the whole noise. The right error is the one against a clean reference, or a criterion that accounts for the expected noise.

## Interview question

<details><summary>How can you use PCA for denoising and how do you choose how many components to keep?</summary>

I reconstruct the data with the first k components, which contain the correlated signal while the noise, spread over all the directions, is mostly discarded. I choose k with a clean reference if I have one, or by comparing the eigenvalues with the expected noise level
(Marchenko-Pastur edge), or with cross-validation on a downstream task. It works if the signal is low-dimensional and the noise is uncorrelated; with correlated noise or a nonlinear signal you need something else (autoencoder, kernel PCA).

</details>